<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Deep-Learning-with-Python/blob/main/scripts/Word_Embedding_Sequence_to_Sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")



from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [2]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Finally, we create our prompt as a user and give it to the model:

In [3]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Why did the chicken join the band? Because it had the drumsticks!


In [4]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"
generator(prompt)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Subject: Heartfelt Apologies for the Gardening Mishap\n\n\nDear Sarah,\n\n\nI hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.\n\n\nAs you know, I have always admired the serene beauty of your garden and the dedication you put into maintaining it. It was with great regret that I accidentally damaged some of your prized rose bushes while attempting to install a new irrigation system.\n\n\nThe mishap happened when I miscalculated the placement of the pipes, which led to a small section of the rose bushes being uprooted. I understand how much time and effort you invest in your garden, and I am truly sorry for the distress this has caused you.\n\n\nPlease know that I am taking full responsibility for the damage. I am committed to restoring your garden to its former glory and will be personally overseeing the replanting process. I will also cover all the costs associated w

In [5]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)

# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


In [6]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


In [7]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [8]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')

In [9]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


### **Tokenization**

In [26]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [27]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [28]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

In [29]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

In [30]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

### **Contextualized Word Embeddings From a Language Model (Like BERT)**

In [10]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  241MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

In [13]:
output

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

In [11]:
output.shape

torch.Size([1, 4, 384])

In [12]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


### **Text Embeddings (For Sentences and Whole Documents)**

In [14]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
vector

array([-2.02203933e-02,  4.57696877e-02, -1.26637146e-02, -3.37991631e-03,
       -2.54910300e-03, -3.31096235e-03, -4.58366945e-02,  2.32390799e-02,
       -3.12585123e-02, -3.15588824e-02, -2.19783355e-02,  3.67821753e-03,
        3.39105981e-03, -1.88130587e-02, -2.65821870e-02, -1.45892315e-02,
        3.20066400e-02, -3.66493082e-03,  1.75410733e-02,  5.43209612e-02,
       -3.96661647e-02,  9.93637647e-03, -3.27205285e-02, -1.62909944e-02,
        6.36525359e-03, -1.24459760e-02,  1.04737049e-02,  3.08674313e-02,
        7.47453189e-03, -3.86319868e-02, -6.19924963e-02, -3.25308628e-02,
       -1.40268244e-02,  5.16586639e-02,  1.69651707e-06, -1.74092929e-04,
       -1.27881172e-03, -4.15052772e-02,  5.00347884e-03,  4.58600894e-02,
       -2.85337027e-02,  5.02370670e-02, -1.75183658e-02, -3.98830482e-04,
        1.63311344e-02,  5.89936748e-02, -2.21730098e-02, -4.00975086e-02,
       -6.60439581e-03, -2.97632981e-02, -3.18221636e-02, -1.77945811e-02,
       -1.54363075e-02, -

In [16]:
vector.shape

(768,)

### **Word Embeddings Beyond LLMs**

In [18]:
!pip install gensim
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 24.1 MB/s eta 0:00:00
[==================================================] 100.0% 66.0/66.0MB downloaded


In [19]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

### **1. transformers (Hugging Face)를 활용한 방법**

- 토큰화 과정과 임베딩 벡터 생성 과정

In [22]:
import torch
from transformers import AutoModel, AutoTokenizer

# 1. 모델 및 토크나이저 명시적 로드 (이전 Gensim model 변수와 충돌 방지)
hf_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
hf_model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# 입력 텍스트
text = "자연어 처리는 정말 재미있습니다."

# 2. 토큰화 (Tokenization)
tokens = hf_tokenizer.tokenize(text)
inputs = hf_tokenizer(text, return_tensors="pt")

print("--- 토큰화 결과 ---")
print("토큰 단어:", tokens)
print("토큰 ID (Input IDs):", inputs["input_ids"])

# 3. 임베딩 벡터 추출
with torch.no_grad():
    outputs = hf_model(**inputs)

# outputs.last_hidden_state: (batch_size, sequence_length, hidden_dimension)
token_embeddings = outputs.last_hidden_state

# Mean Pooling: 각 토큰 벡터들의 평균을 구하여 문장 대표 임베딩 생성
attention_mask = inputs["attention_mask"].unsqueeze(-1)
sentence_embedding = torch.sum(token_embeddings * attention_mask, dim=1) / torch.clamp(attention_mask.sum(dim=1), min=1e-9)

print("\n--- 임베딩 결과 ---")
print("문장 임베딩 차원:", sentence_embedding.shape) # (1, 384)
print("문장 임베딩 벡터 일부:", sentence_embedding[0][:5])

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

--- 토큰화 결과 ---
토큰 단어: ['ì', 'ŀ', 'Ĳ', 'ì', 'Ĺ', '°', 'ì', 'ĸ', '´', 'Ġì', '²', 'ĺ', 'ë', '¦', '¬', 'ë', 'Ĭ', 'Ķ', 'Ġì', 'ł', 'ķ', 'ë', '§', 'Ĳ', 'Ġì', 'ŀ', '¬', 'ë', '¯', '¸', 'ì', 'ŀ', 'Ī', 'ì', 'Ĭ', 'µ', 'ëĭ', 'Ī', 'ëĭ', '¤', '.']
토큰 ID (Input IDs): tensor([[    1, 43998, 17772, 16948, 43998,  6800,  7487, 43998, 25448, 20024,
         46747, 14292,   711, 45209, 18164, 11582, 45209, 27969, 10674, 46747,
         21402, 15722, 45209,  6248, 16948, 46747, 17772, 11582, 45209, 10965,
         18537, 43998, 17772, 23133, 43998, 27969,  8906, 48444, 23133, 48444,
         10470,     4,     2]])

--- 임베딩 결과 ---
문장 임베딩 차원: torch.Size([1, 384])
문장 임베딩 벡터 일부: tensor([-0.8837,  0.3681,  0.2776, -1.7093, -0.2356])


### **2. sentence-transformers를 활용한 간편한 방법**

In [24]:
from sentence_transformers import SentenceTransformer

# 문장 임베딩 전용 모델 로드 (공개적으로 사용 가능한 한국어 SBERT 모델로 변경)
model = SentenceTransformer("jhgan/ko-sbert-sts")

# 입력 텍스트
sentences = ["자연어 처리는 재미있습니다.", "텍스트 임베딩을 생성합니다."]

# 임베딩 생성 (토크나이징 및 풀링 자동 수행)
embeddings = model.encode(sentences)

print("임베딩 결과 Shape:", embeddings.shape) # (2, 768)
print("첫 번째 문장 임베딩 일부:", embeddings[0][:5])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.44k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 결과 Shape: (2, 768)
첫 번째 문장 임베딩 일부: [0.5069231  1.0248408  0.22269492 2.056479   0.06332443]


### **3. text --> onehotcode --> embedding**

In [25]:
import torch
import torch.nn as nn

# 1. 텍스트 및 어휘 사전(Vocabulary) 정의
text = "자연어 처리 및 임베딩 파이프라인"
tokens = text.split()  # 간단한 단어 단위 토큰화

# 토큰-인덱스 매핑 사전 구축
vocab = {token: i for i, token in enumerate(sorted(set(tokens)))}
vocab_size = len(vocab)
embedding_dim = 4  # 임베딩 차원 (예: 4차원)

print("--- 1. 토큰화 및 사전 구축 ---")
print("토큰 목록:", tokens)
print("어휘 사전 (Vocab):", vocab)

# 2. 토큰을 인덱스로 변환
token_ids = [vocab[token] for token in tokens]
input_tensor = torch.tensor(token_ids)

# 3. 원-핫 인코딩 (One-Hot Encoding)
# shape: (sequence_length, vocab_size)
one_hot = torch.nn.functional.one_hot(input_tensor, num_classes=vocab_size).float()

print("\n--- 2. 원-핫 인코딩 (One-Hot Encoded Matrix) ---")
print(one_hot)

# 4. 임베딩 레이어 정의 (가중치 행렬 W: vocab_size x embedding_dim)
# nn.Linear(bias=False)를 사용하여 원-핫 벡터와 행렬 곱을 직접 수행
embedding_layer = nn.Linear(vocab_size, embedding_dim, bias=False)

# 5. 원-핫 벡터 -> 임베딩 벡터 변환 (행렬 곱: One-Hot × Weight)
dense_embeddings = embedding_layer(one_hot)

print("\n--- 3. 최종 임베딩 결과 (Dense Embeddings) ---")
print("임베딩 결과 Shape:", dense_embeddings.shape)  # (sequence_length, embedding_dim)
print(dense_embeddings)

--- 1. 토큰화 및 사전 구축 ---
토큰 목록: ['자연어', '처리', '및', '임베딩', '파이프라인']
어휘 사전 (Vocab): {'및': 0, '임베딩': 1, '자연어': 2, '처리': 3, '파이프라인': 4}

--- 2. 원-핫 인코딩 (One-Hot Encoded Matrix) ---
tensor([[0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1.]])

--- 3. 최종 임베딩 결과 (Dense Embeddings) ---
임베딩 결과 Shape: torch.Size([5, 4])
tensor([[-0.0590,  0.0556, -0.2145,  0.3231],
        [-0.0680, -0.2348, -0.0598,  0.2616],
        [-0.3702,  0.1452,  0.0772,  0.2852],
        [ 0.3643, -0.1445,  0.1826, -0.0576],
        [ 0.3423, -0.1087,  0.3654, -0.3212]], grad_fn=<MmBackward0>)


### **4. Word2Vec**
- 2013년 구글(Google)의 토마스 미콜로프(Tomas Mikolov) 연구팀이 제안한 단어를 밀집 벡터(Dense Vector) 형태로 변환하는 대표적인 대표적 워드 임베딩(Word Embedding) 기법

> **CBOW (Continuous Bag-of-Words)**

>> - 주변 단어들(Context Words)을 가지고 중심 단어(Target Word)를 예측하는 방식

>> - 예: [ "오늘", "날씨가", "??", "좋다" ] $\rightarrow$ ?에 들어갈 단어 "매우" 예측특징: 학습 속도가 빠르며, 자주 등장하는 단어 예측에 효과적

> **Skip-Gram중심 단어(Target Word)**

>> - 주변 단어들(Context Words)을 예측하는 방식

>> - 예: [ "?", "?", "매우", "?", "?" ] $\rightarrow$ 중심 단어 "매우"를 기준으로 주변 단어 예측특징: CBOW보다 학습량이 많지만, 저빈도 단어(드물게 나오는 단어)에 대한 임베딩 품질이 더 우수하여 실무에서 주로 사용

### **Seq2Seq(Sequence-to-Sequence) 모델**

In [32]:
from gensim.models import Word2Vec

# 1. 토큰화된 문장 데이터 준비
sentences = [
    ["자연어", "처리", "딥러닝", "임베딩"],
    ["Word2Vec", "단어", "임베딩", "벡터"],
    ["딥러닝", "인공지능", "모델", "학습"],
    ["왕", "남자", "여왕", "여자"]
]

# 2. Word2Vec 모델 학습 (Skip-Gram 방식: sg=1)
model = Word2Vec(
    sentences=sentences,
    vector_size=100,  # 임베딩 차원
    window=2,         # 문맥 윈도우 크기
    min_count=1,      # 최소 단어 빈도수
    sg=1              # 1: Skip-Gram, 0: CBOW
)

# 3. 단어 벡터 추출
print("--- '단어'의 임베딩 벡터 (크기: 100) ---")
print(model.wv["단어"][:5]) # 일부 5개 값 출력

# 4. 단어 유사도 계산
similarity = model.wv.similarity("단어", "임베딩")
print(f"\n'단어'와 '임베딩'의 유사도: {similarity:.4f}")

# 5. 가장 유사한 단어 탑 N개 찾기
print("\n--- '딥러닝'과 가장 유사한 단어들 ---")
print(model.wv.most_similar("딥러닝", topn=2))

--- '단어'의 임베딩 벡터 (크기: 100) ---
[ 0.0070888  -0.00156793  0.0079475  -0.00948866 -0.0080295 ]

'단어'와 '임베딩'의 유사도: 0.0629

--- '딥러닝'과 가장 유사한 단어들 ---
[('자연어', 0.13725273311138153), ('여왕', 0.06797593832015991)]


Word2Vec의 한계점
- 동음이의어 처리 불가 (Context-independent):

> - "배(Fruit)"와 "배(Boat)"처럼 완전히 다른 의미를 가져도 한 단어당 단 하나의 고정된 벡터

- OOV (Out-of-Vocabulary) 문제:

> - 학습할 때 어휘 사전에 없던 새로운 단어가 입력되면 벡터를 얻을 수 없음

💡 **이러한 한계점을 보완하기 위해 등장한 것이 FastText(철자 단위 학습으로 OOV 해결)와 BERT / GPT(문맥에 따라 단어 벡터가 변하는 트랜스포머 기반 모델)**

In [33]:
import torch
import torch.nn as nn

# 1. 하이퍼파라미터 및 가상 데이터 설정
input_dim = 10   # 입력 어휘 사전 크기 (Vocab Size)
output_dim = 10  # 출력 어휘 사전 크기
emb_dim = 16     # 임베딩 차원
hidden_dim = 32  # RNN Hidden State 차원

# [Batch Size = 2, Sequence Length = 5]
src = torch.randint(0, input_dim, (2, 5))  # 입력 문장 토큰 IDs
trg = torch.randint(0, output_dim, (2, 5))  # 타겟 문장 토큰 IDs

# 2. 인코더 (Encoder)
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        # src: (batch_size, seq_len)
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        # hidden: (1, batch_size, hidden_dim) -> 문맥 벡터(Context Vector)
        return hidden

# 3. 디코더 (Decoder)
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, trg, hidden):
        # trg: (batch_size, seq_len)
        embedded = self.embedding(trg)
        # 인코더의 마지막 hidden state를 초기값으로 전달
        outputs, hidden = self.rnn(embedded, hidden)
        predictions = self.fc(outputs)
        # predictions: (batch_size, seq_len, output_dim)
        return predictions

# 4. Seq2Seq 모델 결합
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        hidden = self.encoder(src)
        output = self.decoder(trg, hidden)
        return output

# 5. 모델 생성 및 순전파 테스트
encoder = Encoder(input_dim, emb_dim, hidden_dim)
decoder = Decoder(output_dim, emb_dim, hidden_dim)
model = Seq2Seq(encoder, decoder)

output = model(src, trg)

print("입력 Shape (src):", src.shape)      # torch.Size([2, 5])
print("출력 Shape (output):", output.shape)  # torch.Size([2, 5, 10]) (Batch, Seq_Len, Output_Vocab_Size)

입력 Shape (src): torch.Size([2, 5])
출력 Shape (output): torch.Size([2, 5, 10])
